# Uncertainties, Composition, and the Habitable Zone

Written By: Aiden Zelakiewicz (asz39@cornell.edu) & Abigail Bohl (acb338@cornell.edu)

In this notebook we will learn how to propagate uncertainties, plot error bars, determine which planets are in the Habitable Zone, and discover exoplanet composition. By the end of this notebook, you will have all the tools necessary to complete your homework!

When you see a "<---TO DO--->" header in this notebook, that is an exercise for you to complete! If you are struggling and have questions, ask a neighbor or your instructors for help and we will lead you in the right direction :D

## Goals for this exercise: 

1. Outline error propagation and learn how to 

2. Plot error bars and show which planets are in the Habitable Zone

3. Answer "What is the composition of planets?"

## 1. Error Propagation

### An Brief Introduction to Uncertainty

When we measure a quantity, we often have some uncertainty in that measurement. Uncertainty can come from counting statistics, instrument characteristics, other uncertain physical quantities, or other sources. For example, if we measure the radius of a planet to be 1.5 Earth radii with an uncertainty of 0.1 Earth radii, we would write this as:
$$
R = 1.5 \pm 0.1~R_\oplus
$$
This assumes the radius is normally distributed, such that the true value of the radius can be described by a Gaussian distribution with a mean of $\mu=1.5~R_\oplus$ and a standard deviation of $\sigma=0.1~R_\oplus$. 
To visualize this, let's sample a Gaussian distribution $\mathcal{N}(\mu, \sigma)$ with $\mu=1.5$ and $\sigma=0.1$ and plot the results.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# Create a Gaussian distribution for the radius of the planet
mu = 1.5  # mean radius in Earth radii
sigma = 0.1  # standard deviation in Earth radii

# Calculate the Gaussian (normal) distribution sampling 1 million points
radii = np.random.normal(loc=mu, scale=sigma, size=1000000)

# Plot the histogram of the sampled radii
plt.hist(radii, bins=50, density=True, alpha=0.6, color='b') # using plt.hist for the histogram

# Plot vertical lines (axvline) for the mean and one standard deviation above and below the mean
plt.axvline(mu, color='k', linestyle='dashed', linewidth=2)
plt.axvline(mu + sigma, color='r', linestyle='dashed', linewidth=2)
plt.axvline(mu - sigma, color='r', linestyle='dashed', linewidth=2)

plt.legend([r'$\mu=1.5~R_\oplus$', r'$\sigma=0.1~R_\oplus$'])

plt.xlabel('Radius (Earth Radii)')

plt.show()

This is generally what we mean when we say "uncertainty" in a measurement.
The uncertainty describes the probability distribution of the quantity we are measuring.
When we do calculations with quantities that have uncertainties, we need to propagate those uncertainties through the calculation to determine the uncertainty in the final result.
Not all measurements are normally distributed though, especially in astronomy where Monte Carlo or Nested Sampling simulations are used to determine the posterior distribution of parameters (don't worry about this but feel free to ask Aiden later).

For example, we might not know if a planet is has a larger radius and small atmospher or a smaller radius and a large atmosphere. We might know, however, that the radius more probable to be smaller than larger from measurements, so the distribution of the radius is not symmetric and is not Gaussian. The uncertainty in the radius might instead be
$$
R = 1.5^{+0.1}_{-0.3}~R_\oplus
$$
This is how much of the errors will be reported in the NASA Exoplanet Archive. For a given parameter, `pl_rade` for example, the associated upper (+) and lower (-) uncertainties are reported in the `pl_radeerr1` and `pl_radeerr2` columns, respectively. The upper uncertainty is the difference between the 84th percentile and the median of the posterior distribution, while the lower uncertainty is the difference between the median and the 16th percentile of the posterior distribution.

### Error Propagation

Assume we have a function $f(x, y)$ that depends on two measured quantities $x$ and $y$, each with their own uncertainties $\sigma_x$ and $\sigma_y$. If we assume that the uncertainties are independent and normally distributed, we can propagate the uncertainties through the function using the following formula:
$$
\sigma_f = \sqrt{\left(\frac{\partial f}{\partial x}\right)^2 \sigma_x^2 + \left(\frac{\partial f}{\partial y}\right)^2 \sigma_y^2}
$$
This formula is derived from the Taylor expansion of the function $f(x, y)$ around the mean values of $x$ and $y$. In the case of variable that are not independent, we would need to include a covariance term in the error propagation formula. Additionally, if the uncertainties are not normally distributed, we would need to use a different method to propagate the uncertainties, such as Monte Carlo simulations. That is beyond the scope of this notebook, and sometimes impossible to do with the data available.

Instead, what we will do is something that is commonly done but not strictly correct. We will assume that the Gaussian is split at the median, and that the upper and lower uncertainties are independent. We will then propagate the upper and lower uncertainties separately through the function $f(x, y)$ using the same formula as above. This is not strictly correct, but it is a reasonable approximation for many cases. So for our function $f(x, y)$, we will have two propagated uncertainties, $\sigma_{f}^+$ and $\sigma_{f}^-$, given by:
$$
\sigma_{f}^+ = \sqrt{\left(\frac{\partial f}{\partial x} \sigma_x^+\right)^2 + \left(\frac{\partial f}{\partial y} \sigma_y^+\right)^2}
$$
$$
\sigma_{f}^- = \sqrt{\left(\frac{\partial f}{\partial x} \sigma_x^-\right)^2 + \left(\frac{\partial f}{\partial y} \sigma_y^-\right)^2}
$$

### <---TO DO--->

Warm up time! The equation for the effective stellar flux received by a planet is given by:
$$
S_\text{eff} = \left(\frac{R_\star}{R_\odot}\right)^2 \left(\frac{T_\star}{T_\odot}\right)^4 \left(\frac{d_p}{1~\text{AU}}\right)^{-2}
$$
Where $R_\star$ is the radius of the star, $T_\star$ is the effective temperature of the star, and $d_p$ is the distance of the planet from the star.
Calculated the derivatives of $S_\text{eff}$ with respect to $R_\star$, $T_\star$, and $d_p$. We will use these derivatives to propagate the uncertainties in $R_\star$, $T_\star$, and $d_p$ through the equation for $S_\text{eff}$.

## 2. Planets in the Habitable Zone

Let's move onto using the effective stellar flux to determine which planets are in the Habitable Zone. As of last class, we now have all the tools we need to do this; except we need errors on the effective stellar flux!

Let's get our data read in and recompute the effective stellar flux and stellar luminosity.

In [ ]:
import pandas as pd

# Filename and extension
filename = "<PLACE NAME OF FILE HERE>"

file_path = 'data/' + filename

nea_df = pd.read_csv(file_path, sep=',', comment='#')

In [ ]:
def seff(R_star, T_star, d_p):
    """
    Calculate the effective stellar flux (S_eff) for a planet.
    
    Parameters:
    R_star : float
        Radius of the star in solar radii.
    T_star : float
        Effective temperature of the star in Kelvin.
    d_p : float
        Distance of the planet from the star in AU.
        
    Returns:
    S_eff : float
        Effective stellar flux received by the planet.
    """
    return (R_star**2) * (T_star / 5778)**4 * (1 / d_p**2)

def luminosity(R_star, T_star):
    """
    Calculate the luminosity of a star.
    
    Parameters:
    R_star : float
        Radius of the star in solar radii.
    T_star : float
        Effective temperature of the star in Kelvin.
        
    Returns:
    L_star : float
        Luminosity of the star in solar luminosities.
    """
    return (R_star**2) * (T_star / 5778)**4

Just earlier we derived the derivative of $S_\text{eff}$ with respect to $R_\star$, $T_\star$, and $d_p$. Let's get these into our code so we can propagate the uncertainties in $R_\star$, $T_\star$, and $d_p$ through the equation for $S_\text{eff}$.

In [ ]:
nea_df['S_eff'] = seff(nea_df['R_star'], nea_df['T_star'], nea_df['d_p'])
nea_df['L_star'] = luminosity(nea_df['R_star'], nea_df['T_star'])

# Let's write each of the derivatives in terms of our NEA data
dS_dR = 2 * nea_df['R_star'] * (nea_df['T_star'] / 5778)**4  * nea_df['d_p']**-2
dS_dT = 4 * nea_df['R_star']**2 * (nea_df['T_star'] / 5778)**3 * (1 / 5778) * nea_df['d_p']**-2
dS_dd = -2 * nea_df['R_star']**2 * (nea_df['T_star'] / 5778)**4 * nea_df['d_p']**-3

With our derivatives in hand, we can propagate the uncertainties in $R_\star$, $T_\star$, and $d_p$ through the equation for $S_\text{eff}$ using the error propagation formula we derived earlier.

In [ ]:
# Error propagation formula for S_eff (upper)
sigma_S_plus = np.sqrt((dS_dR * nea_df['st_raderr1'])**2 + (dS_dT * nea_df['st_efferr1'])**2 + (dS_dd * nea_df['pl_orbsmaxerr1'])**2)
sigma_S_minus = np.sqrt((dS_dR * nea_df['st_raderr2'])**2 + (dS_dT * nea_df['st_efferr2'])**2 + (dS_dd * nea_df['pl_orbsmaxerr2'])**2)

# Add the propagated uncertainties to the dataframe
nea_df['S_eff_err_plus'] = sigma_S_plus
nea_df['S_eff_err_minus'] = sigma_S_minus

Great, now that we have the propagated uncertainties in $S_\text{eff}$, we plot the effective stellar flux with error bars!